# Larger Qwen2.5 on the Test Set (Colab, self-contained)

Runs `qwen2.5:32b` and then `qwen2.5:72b` (generation and candidate selection) on the frozen
`data/splits/test_items.csv`, with the same prompts, shuffle seed, scorer and default
decoding as the `qwen2.5:7b` run. Only the model size changes.

Same pattern as `run_test_eval_colab.ipynb`: no repo clone and no `pip install -e .`;
the few small source files are fetched from GitHub and imported as loose modules.
Ollama is installed inside the Colab session.

**Runtime:** the 72B (~47 GB) needs an 80 GB GPU (A100 80 GB or H100); the 32B fits there too.
To run only the 32B, shorten `MODELS` below; an A100 40 GB is enough for that.

Per-item CSVs are written to `qwen32b/` and `qwen72b/` (`test_{select,generate}_details.csv`),
saved after each mode and downloaded as a zip per model. Re-running skips finished modes.

## 1. Fetch source files

In [ ]:
!pip install -q requests tqdm

In [ ]:
import urllib.request, sys, types, importlib.util

REPO_RAW = 'https://raw.githubusercontent.com/BenCarmel123/hebrew-acronym-disambiguation/main'

# dependency order: csv_io -> pairs -> eval -> qwen.eval
FILES = {
    'hebrew_acronyms.data_processing.common.csv_io': 'src/hebrew_acronyms/data_processing/common/csv_io.py',
    'hebrew_acronyms.models.common.pairs': 'src/hebrew_acronyms/models/common/pairs.py',
    'hebrew_acronyms.models.common.eval': 'src/hebrew_acronyms/models/common/eval.py',
    'hebrew_acronyms.models.qwen.eval': 'src/hebrew_acronyms/models/qwen/eval.py',
}

# empty package stubs so the dotted imports resolve without installing the package
pkgs = {'.'.join(d.split('.')[:i]) for d in FILES for i in range(1, len(d.split('.')))}
for pkg in sorted(pkgs, key=len):
    mod = types.ModuleType(pkg)
    mod.__path__ = []
    sys.modules[pkg] = mod

def fetch_module(dotted_name, repo_path):
    local_path = dotted_name.replace('.', '_') + '.py'
    urllib.request.urlretrieve(f'{REPO_RAW}/{repo_path}', local_path)
    spec = importlib.util.spec_from_file_location(dotted_name, local_path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[dotted_name] = mod
    spec.loader.exec_module(mod)

for dotted, path in FILES.items():
    fetch_module(dotted, path)

urllib.request.urlretrieve(f'{REPO_RAW}/data/splits/test_items.csv', 'test_items.csv')
print('modules and test_items.csv fetched')

## 2. Install and start Ollama

`zstd` is needed by the Ollama installer. If Colab asks for a runtime restart, restart and
re-run from the top; nothing here depends on state surviving a restart.

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)
print('ollama server started, pid', ollama_proc.pid)

## 3. Pull and run each model

Models run one after the other: pull, select, generate, save, download, then free the model.

In [ ]:
MODELS = ['qwen2.5:32b', 'qwen2.5:72b']
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 4. Run the test set

In [ ]:
import os, subprocess
from tqdm.auto import tqdm
import hebrew_acronyms.models.common.eval as eval_module
from hebrew_acronyms.models.common.eval import evaluate, write_details_csv
from hebrew_acronyms.models.common.pairs import load_rows
from hebrew_acronyms.models.qwen.eval import ollama_generate, ollama_model_identity
from google.colab import files

rows = load_rows('test_items.csv')
eval_module.tqdm = lambda iterable, **kwargs: iterable   # silence the evaluator's own bar; one bar per model and mode below

for model in tqdm(MODELS, desc='models', unit='model'):
    subprocess.run(['ollama', 'pull', model], check=True)
    print(model, 'digest:', ollama_model_identity(model)['digest'])
    tag = model.replace('qwen2.5:', 'qwen')
    os.makedirs(tag, exist_ok=True)
    for mode in ('select', 'generate'):
        out = f'{tag}/test_{mode}_details.csv'
        if os.path.exists(out):
            print(model, mode, 'already saved, skipping')
            continue
        with tqdm(total=len(rows), desc=f'{model} {mode}', unit='item') as bar:
            def generate_fn(prompt):
                answer = ollama_generate(prompt, model=model, timeout=600)
                bar.update(1)
                return answer
            res = evaluate(rows, generate_fn=generate_fn, mode=mode)
        print(f"{model} {mode}: accuracy {res['accuracy']:.3f}  invalid {res['invalid_rate']:.3f}  n {res['n_items']}")
        write_details_csv(out, res['details'])
    subprocess.run(['zip', '-qr', f'{tag}.zip', tag], check=True)
    files.download(f'{tag}.zip')
    subprocess.run(['ollama', 'stop', model])   # unload from GPU before the next model
    subprocess.run(['ollama', 'rm', model])     # free disk